In [ ]:
!pip install -q "pillow<13.0" roboflow ultralytics huggingface_hub



In [ ]:
import os
from roboflow import Roboflow

# 1. Lấy API Key (Ưu tiên Kaggle Secrets, nếu không có sẽ lấy chuỗi dán trực tiếp)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("ROBOFLOW_API_KEY")
except Exception:
    # Thay thế bằng API Key thực tế lấy từ https://app.roboflow.com/settings/api
    api_key = "NHAP_ROBOFLOW_API_KEY_CUA_BAN" 

# 2. Kết nối tới Roboflow và tải dataset phiên bản YOLOv8/YOLO11
rf = Roboflow(api_key=api_key)
project = rf.workspace("giaothongdetectai").project("hcm-traffic-detection-v2")
version = project.version(1) # Thay số version nếu bạn muốn dùng bản khác
dataset = version.download("yolov8")

# 3. Xác nhận đường dẫn file data.yaml
yaml_path = os.path.join(dataset.location, "data.yaml")
print("\n[Thành công] Dataset được tải về tại:", dataset.location)
print("[Thành công] File cấu hình YAML:", yaml_path)

In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình
model = YOLO("yolo11l.pt")

# Huấn luyện trên GPU Kaggle (T4 x2 hoặc P100)
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=1024,
    batch=4,       # Kaggle GPU (16GB VRAM) chạy ổn định ở batch=4 với imgsz=1024
    device=0,
    mosaic=1.0,
    mixup=0.15,
    close_mosaic=10,
    project="/kaggle/working/yolo_hcm_traffic",
    name="run_1"
)

In [ ]:
import shutil
# Copy weights ra ngoài root làm việc sau khi train xong
shutil.copy(results.save_dir + "/weights/best.pt", "/kaggle/working/best.pt")

In [ ]:
from huggingface_hub import HfApi

# Khởi tạo API
api = HfApi(token="NHAP_HUGGINGFACE_API_TOKEN_CUA_BAN")  # Thay bằng token thực tế lấy từ https://huggingface.co/settings/tokens

# Tên repo trên Hugging Face (Thay username_cua_ban bằng username HF của bạn)
repo_id = "username_cua_ban/yolo-hcm-traffic-weights"

# Tự động tạo repo Private nếu chưa có và upload file
api.create_repo(repo_id=repo_id, private=True, exist_ok=True)

api.upload_file(
    path_or_fileobj="/kaggle/working/yolo_hcm_traffic_best.pt",
    path_in_repo="best.pt",
    repo_id=repo_id,
    repo_type="model"
)
print("✅ Đã lưu vĩnh viễn file best.pt lên Hugging Face Hub!")